# 03 - 基金风格报告生成

本 Notebook 用于：
1. 基金收益对 Barra 因子回归
2. 计算风格暴露度
3. 滚动窗口分析（风格漂移）
4. 生成完整报告

In [ ]:
import sys
sys.path.insert(0, '..')

import config
from barra import DataLoader, FactorBuilder, BarraRegression, BarraPlotter

import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

## 1. 加载数据

In [ ]:
loader = DataLoader(config.DATA_DIR, config.START_DATE, config.END_DATE)

# 基金数据
fund_nav = loader.load_fund_nav(config.FUND_CODE)
fund_return = fund_nav['return']

# 因子数据
index_df = loader.load_all_indexes(config.FACTOR_INDEXES)
factors = FactorBuilder(index_df).build()

print(f"基金: {config.FUND_NAME}")
print(f"数据对齐: {len(fund_return)} 个交易日")

## 2. OLS 回归分析

In [ ]:
reg = BarraRegression(fund_return, factors)
reg.fit()

print(reg.summary())

In [ ]:
# 提取暴露度
exposures = reg.exposures()
print("\n风格暴露度:")
for f, v in exposures.items():
    print(f"  {f:12s}: {v:+.4f}")

print(f"\nR² = {reg.results.rsquared:.4f}")
print(f"调整 R² = {reg.results.rsquared_adj:.4f}")

## 3. 基金业绩指标

In [ ]:
perf = BarraRegression.performance_metrics(fund_return)

print("基金业绩表现:")
print(f"  累计收益: {perf['cum_return']*100:+.2f}%")
print(f"  年化收益: {perf['annual_return']*100:+.2f}%")
print(f"  年化波动: {perf['annual_vol']*100:.2f}%")
print(f"  夏普比率: {perf['sharpe']:.2f}")
print(f"  最大回撤: {perf['max_drawdown']*100:.2f}%")

## 4. 滚动窗口分析（风格漂移）

In [ ]:
rolling = reg.rolling_fit(window=60)

print(f"滚动窗口: {len(rolling)} 个")
rolling.head()

In [ ]:
# 滚动暴露统计
print("滚动暴露统计:")
print(rolling[['market', 'size', 'value', 'momentum']].describe())

## 5. 生成完整报告图

In [ ]:
plotter = BarraPlotter(dpi=150)

fig = plotter.plot_report(
    fund_nav=(1 + fund_return).cumprod(),
    exposures=exposures,
    rolling=rolling,
    output_path=f"{config.OUTPUT_DIR}/report_{config.FUND_CODE}.png"
)
plt.show()

## 6. 导出结果

In [ ]:
# 导出 CSV
rolling.to_csv(f"{config.OUTPUT_DIR}/rolling_exposures_{config.FUND_CODE}.csv")
exposures.to_frame('exposure').to_csv(f"{config.OUTPUT_DIR}/exposures_{config.FUND_CODE}.csv")

# 生成文本报告
report = f"""
Barra 风格分析报告
==================
基金: {config.FUND_NAME} ({config.FUND_CODE})
日期: {datetime.now().strftime('%Y-%m-%d %H:%M')}

业绩表现:
  累计收益: {perf['cum_return']*100:+.2f}%
  年化收益: {perf['annual_return']*100:+.2f}%
  夏普比率: {perf['sharpe']:.2f}
  最大回撤: {perf['max_drawdown']*100:.2f}%

风格暴露:
"""
for f, v in exposures.items():
    report += f"  {f}: {v:+.4f}\n"
report += f"\nR² = {reg.results.rsquared:.4f}"

with open(f"{config.OUTPUT_DIR}/report_{config.FUND_CODE}.txt", 'w') as f:
    f.write(report)

print("✓ 报告导出完成")